In [1]:
# ============================================
# TEST PIPELINE ON FRESH DATA
# ============================================

import pandas as pd
import requests
import time
import sys

# Add src to path
sys.path.append('../')

from src.pipeline import preprocess_and_engineer_features, save_cleaned_data

# ============================================
# 1. FETCH FRESH RAW DATA
# ============================================

def fetch_coin_info(coin_id):
    """Fetch coin metadata from CoinGecko"""
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"Failed to fetch info for {coin_id}")
    
    data = response.json()
    
    return {
        "coin": coin_id,
        "market_cap_rank": data["market_cap_rank"],
        "circulating_supply": data["market_data"]["circulating_supply"],
        "total_supply": data["market_data"]["total_supply"],
        "max_supply": data["market_data"]["max_supply"],
        "ath": data["market_data"]["ath"]["usd"],
        "atl": data["market_data"]["atl"]["usd"],
    }

def fetch_coin_history(coin_id, days=365):
    """Fetch historical price data from CoinGecko"""
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {
        "vs_currency": "usd",
        "days": days
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        raise Exception(
            f"Failed to fetch history for {coin_id} | "
            f"Status: {response.status_code} | Response: {response.text}"
        )
    
    data = response.json()
    
    df = pd.DataFrame({
        "timestamp": [x[0] for x in data["prices"]],
        "price": [x[1] for x in data["prices"]],
        "market_cap": [x[1] for x in data["market_caps"]],
        "volume": [x[1] for x in data["total_volumes"]],
    })
    
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["coin"] = coin_id
    
    return df

# ============================================
# 2. LOAD FRESH DATA
# ============================================

print("📥 Fetching fresh data from CoinGecko API...")

COINS = ["bitcoin", "ethereum"]
DAYS = 365

all_data = []

for coin in COINS:
    print(f"  ├─ Loading {coin}...")
    
    # Fetch history
    history_df = fetch_coin_history(coin, DAYS)
    
    # Fetch metadata
    info_dict = fetch_coin_info(coin)
    
    # Merge metadata into history
    for key, value in info_dict.items():
        history_df[key] = value
    
    all_data.append(history_df)
    time.sleep(1)  # Be nice to API

# Combine
raw_df = pd.concat(all_data, ignore_index=True)

print(f"  └─ Loaded {len(raw_df)} rows\n")
print("Raw data shape:", raw_df.shape)
print("Raw data columns:", raw_df.columns.tolist())
print("\nRaw data sample:")
print(raw_df.head())

# ============================================
# 3. TEST PIPELINE
# ============================================

print("\n" + "="*60)
print("🧪 TESTING PIPELINE")
print("="*60 + "\n")

# Run pipeline
clean_df = preprocess_and_engineer_features(raw_df)

# ============================================
# 4. VERIFY OUTPUT
# ============================================

print("\n" + "="*60)
print("✅ VERIFICATION")
print("="*60 + "\n")

print(f"Output shape: {clean_df.shape}")
print(f"Expected columns: 26")
print(f"Actual columns: {len(clean_df.columns)}\n")

print("Columns created:")
print(clean_df.columns.tolist())

print("\nFirst 5 rows:")
print(clean_df.head())

print("\nLast 5 rows:")
print(clean_df.tail())

print("\nData types:")
print(clean_df.dtypes)

print("\nNaN check:")
print(clean_df.isna().sum().sum(), "NaNs found")

# ============================================
# 5. SAVE CLEANED DATA
# ============================================

print("\n" + "="*60)
print("💾 SAVING CLEANED DATA")
print("="*60 + "\n")

output_path = save_cleaned_data(clean_df, output_path="../data/processed")

print(f"\n✅ Pipeline test complete!")
print(f"📁 Cleaned data saved to: {output_path}")

📥 Fetching fresh data from CoinGecko API...
  ├─ Loading bitcoin...
  ├─ Loading ethereum...
  └─ Loaded 732 rows

Raw data shape: (732, 11)
Raw data columns: ['timestamp', 'price', 'market_cap', 'volume', 'coin', 'market_cap_rank', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'atl']

Raw data sample:
   timestamp         price    market_cap        volume     coin  \
0 2025-02-12  95739.977371  1.898789e+12  3.645458e+10  bitcoin   
1 2025-02-13  97836.188561  1.936842e+12  4.711562e+10  bitcoin   
2 2025-02-14  96561.663999  1.914315e+12  2.908437e+10  bitcoin   
3 2025-02-15  97488.481485  1.931657e+12  3.228991e+10  bitcoin   
4 2025-02-16  97569.951694  1.934308e+12  1.421531e+10  bitcoin   

   market_cap_rank  circulating_supply  total_supply  max_supply       ath  \
0                1          19987621.0    19987621.0  21000000.0  126080.0   
1                1          19987621.0    19987621.0  21000000.0  126080.0   
2                1          19987621.0    1998

C:\Users\tlili\AppData\Local\Temp\ipykernel_14560\72296777.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat(all_data, ignore_index=True)
